# Word Embeddings: Visual Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/methods-algorithms/blob/master/notebooks/L06_embeddings.ipynb)

**Learning Objectives:**
- Understand how word embeddings map words to dense vectors where similar words are close
- Visualize embedding spaces and measure similarity with cosine distance
- Perform word analogy tasks using vector arithmetic
- Use TF-IDF and SVD as a simple embedding pipeline for text classification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.manifold import TSNE

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ML color palette
ML_PURPLE = '#3333B2'
ML_BLUE = '#0066CC'
ML_ORANGE = '#FF7F0E'
ML_GREEN = '#2CA02C'
ML_RED = '#D62728'
ML_LAVENDER = '#ADADE0'

CLUSTER_COLORS = [ML_GREEN, ML_RED, ML_BLUE]
CATEGORY_COLORS = [ML_BLUE, ML_ORANGE, ML_GREEN, ML_RED]

print('Setup complete.')

## 1. One-Hot vs Dense Embeddings

In [ ]:
# 10 finance words
words_10 = ['stock', 'bond', 'equity', 'debt', 'rally',
            'crash', 'profit', 'loss', 'bull', 'bear']
n_words = len(words_10)

# One-hot encoding: identity matrix
one_hot = np.eye(n_words)

# Dense embedding: random 3D vectors (for illustration)
dense_emb = np.array([
    [0.9, 0.2, 0.3],   # stock
    [0.1, 0.9, 0.2],   # bond
    [0.85, 0.15, 0.25], # equity
    [0.15, 0.85, 0.3],  # debt
    [0.7, 0.1, 0.8],   # rally
    [0.7, 0.1, -0.7],  # crash
    [0.6, 0.3, 0.7],   # profit
    [0.6, 0.3, -0.6],  # loss
    [0.5, 0.2, 0.9],   # bull
    [0.5, 0.2, -0.8],  # bear
])

# Visual 1: Side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# One-hot
ax = axes[0]
im = ax.imshow(one_hot, cmap='Purples', aspect='auto')
ax.set_xticks(range(n_words))
ax.set_xticklabels(range(1, n_words + 1), fontsize=9)
ax.set_yticks(range(n_words))
ax.set_yticklabels(words_10, fontsize=10)
ax.set_xlabel('Dimension')
ax.set_title('One-Hot Encoding (10D)')
plt.colorbar(im, ax=ax, shrink=0.8)

# Dense embedding
ax = axes[1]
im = ax.imshow(dense_emb, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(3))
ax.set_xticklabels(['Dim 1', 'Dim 2', 'Dim 3'], fontsize=10)
ax.set_yticks(range(n_words))
ax.set_yticklabels(words_10, fontsize=10)
ax.set_xlabel('Dimension')
ax.set_title('Dense Embedding (3D)')
plt.colorbar(im, ax=ax, shrink=0.8)

for i in range(n_words):
    for j in range(3):
        ax.text(j, i, f'{dense_emb[i, j]:.1f}', ha='center', va='center',
                fontsize=8, color='white' if abs(dense_emb[i, j]) > 0.5 else 'black')

fig.suptitle('One-Hot vs Dense Embeddings', fontsize=14)
plt.tight_layout()
plt.show()

print(f'One-hot dimensionality: {one_hot.shape[1]}')
print(f'Dense dimensionality: {dense_emb.shape[1]}')
print(f'One-hot similarity (stock, equity): {cosine_similarity(one_hot[[0]], one_hot[[2]])[0,0]:.3f}')
print(f'Dense similarity (stock, equity): {cosine_similarity(dense_emb[[0]], dense_emb[[2]])[0,0]:.3f}')

## 2. Exploring the Embedding Space

In [ ]:
# 15 finance words in 3 clusters with hand-crafted 3D embeddings
words_15 = [
    # Bullish cluster
    'rally', 'surge', 'upgrade', 'outperform', 'bullish',
    # Bearish cluster
    'crash', 'plunge', 'downgrade', 'underperform', 'bearish',
    # Instruments cluster
    'stock', 'bond', 'equity', 'option', 'futures',
]
cluster_labels = ['Bullish'] * 5 + ['Bearish'] * 5 + ['Instruments'] * 5
cluster_ids = [0] * 5 + [1] * 5 + [2] * 5

embeddings_15 = np.array([
    # Bullish: high x, high z
    [0.8, 0.3, 0.9], [0.9, 0.2, 0.85], [0.75, 0.4, 0.8],
    [0.85, 0.35, 0.75], [0.7, 0.25, 0.95],
    # Bearish: high x, low z
    [0.8, 0.3, -0.85], [0.9, 0.2, -0.9], [0.75, 0.4, -0.75],
    [0.85, 0.35, -0.8], [0.7, 0.25, -0.9],
    # Instruments: low x, mid z
    [0.1, 0.9, 0.1], [0.05, 0.85, -0.1], [0.15, 0.88, 0.05],
    [0.2, 0.8, 0.15], [0.12, 0.82, -0.05],
])

# Visual 2: 3D scatter plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

for cid, (cname, color) in enumerate(zip(['Bullish', 'Bearish', 'Instruments'], CLUSTER_COLORS)):
    mask = np.array(cluster_ids) == cid
    ax.scatter(embeddings_15[mask, 0], embeddings_15[mask, 1], embeddings_15[mask, 2],
               s=100, c=color, label=cname, alpha=0.8, edgecolors='white')

for i, word in enumerate(words_15):
    ax.text(embeddings_15[i, 0] + 0.03, embeddings_15[i, 1] + 0.03,
            embeddings_15[i, 2] + 0.03, word, fontsize=8)

ax.set_xlabel('Dim 1')
ax.set_ylabel('Dim 2')
ax.set_zlabel('Dim 3')
ax.set_title('Word Embedding Space: Finance Terms (3D)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Visual 3: Cosine similarity heatmap
sim_matrix = cosine_similarity(embeddings_15)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(15))
ax.set_xticklabels(words_15, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(15))
ax.set_yticklabels(words_15, fontsize=9)
ax.set_title('Cosine Similarity Between Finance Word Embeddings')
plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.8)

# Add cluster separators
for pos in [4.5, 9.5]:
    ax.axhline(pos, color='black', linewidth=1.5)
    ax.axvline(pos, color='black', linewidth=1.5)

plt.tight_layout()
plt.show()

print(f'Similarity (rally, surge): {sim_matrix[0, 1]:.3f}')
print(f'Similarity (rally, crash): {sim_matrix[0, 5]:.3f}')
print(f'Similarity (stock, bond): {sim_matrix[10, 11]:.3f}')

## 3. Word Analogies

In [ ]:
# Extended vocabulary with analogy-friendly hand-crafted embeddings
analogy_words = ['king', 'queen', 'man', 'woman', 'prince', 'princess',
                 'stock', 'bond', 'equity', 'debt', 'share', 'loan']
analogy_vecs = np.array([
    [0.9, 0.7, 0.1, 0.1],    # king
    [0.9, 0.7, 0.1, 0.85],   # queen
    [0.3, 0.1, 0.1, 0.1],    # man
    [0.3, 0.1, 0.1, 0.85],   # woman
    [0.7, 0.5, 0.1, 0.1],    # prince
    [0.7, 0.5, 0.1, 0.85],   # princess
    [0.1, 0.1, 0.9, 0.2],    # stock
    [0.1, 0.1, 0.15, 0.85],  # bond
    [0.1, 0.1, 0.85, 0.15],  # equity
    [0.1, 0.1, 0.2, 0.9],    # debt
    [0.1, 0.1, 0.88, 0.18],  # share
    [0.1, 0.1, 0.18, 0.88],  # loan
])

def find_nearest(target_vec, vocab_vecs, vocab_words, exclude_indices, top_k=3):
    sims = cosine_similarity(target_vec.reshape(1, -1), vocab_vecs)[0]
    for idx in exclude_indices:
        sims[idx] = -2  # exclude
    top_ids = np.argsort(sims)[::-1][:top_k]
    return [(vocab_words[i], sims[i]) for i in top_ids]

# Analogy 1: king - man + woman = ?
result_1 = analogy_vecs[0] - analogy_vecs[2] + analogy_vecs[3]  # king - man + woman
nearest_1 = find_nearest(result_1, analogy_vecs, analogy_words, exclude_indices=[0, 2, 3])

# Analogy 2: stock - equity + debt = ?
result_2 = analogy_vecs[6] - analogy_vecs[8] + analogy_vecs[9]  # stock - equity + debt
nearest_2 = find_nearest(result_2, analogy_vecs, analogy_words, exclude_indices=[6, 8, 9])

# Visual 4: Bar charts of top-3 nearest neighbors
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Analogy 1
ax = axes[0]
names_1 = [n for n, s in nearest_1]
scores_1 = [s for n, s in nearest_1]
colors_1 = [ML_PURPLE if n == 'queen' else ML_LAVENDER for n in names_1]
bars = ax.barh(names_1[::-1], scores_1[::-1], color=colors_1[::-1], edgecolor='white')
for bar, score in zip(bars, scores_1[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=10)
ax.set_xlabel('Cosine Similarity')
ax.set_title('king - man + woman = ?')
ax.set_xlim(0, 1.15)

# Analogy 2
ax = axes[1]
names_2 = [n for n, s in nearest_2]
scores_2 = [s for n, s in nearest_2]
colors_2 = [ML_ORANGE if n == 'bond' else ML_LAVENDER for n in names_2]
bars = ax.barh(names_2[::-1], scores_2[::-1], color=colors_2[::-1], edgecolor='white')
for bar, score in zip(bars, scores_2[::-1]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{score:.3f}', va='center', fontsize=10)
ax.set_xlabel('Cosine Similarity')
ax.set_title('stock - equity + debt = ?')
ax.set_xlim(0, 1.15)

fig.suptitle('Word Analogy Results: Top-3 Nearest Neighbors', fontsize=14)
plt.tight_layout()
plt.show()

print(f'king - man + woman => {nearest_1[0][0]} (sim={nearest_1[0][1]:.3f})')
print(f'stock - equity + debt => {nearest_2[0][0]} (sim={nearest_2[0][1]:.3f})')

## 4. TF-IDF: A Simple Text Representation

In [ ]:
# Load 20 Newsgroups subset
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.misc']
newsgroups = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes'), random_state=42
)
X_text, y_text = newsgroups.data, newsgroups.target
target_names = newsgroups.target_names

print(f'Documents: {len(X_text)}')
print(f'Categories: {target_names}')
print(f'Distribution: {np.bincount(y_text)}')

# Fit TF-IDF
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_tfidf = tfidf.fit_transform(X_text)
feature_names = np.array(tfidf.get_feature_names_out())

print(f'TF-IDF matrix shape: {X_tfidf.shape}')
print(f'Sparsity: {1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.1%}')

# Visual 5: Top-10 TF-IDF terms per category
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for cat_id, (ax, cat_name, color) in enumerate(
        zip(axes.ravel(), target_names, CATEGORY_COLORS)):
    mask = y_text == cat_id
    mean_tfidf = np.asarray(X_tfidf[mask].mean(axis=0)).ravel()
    top_idx = np.argsort(mean_tfidf)[::-1][:10]
    top_terms = feature_names[top_idx]
    top_scores = mean_tfidf[top_idx]

    ax.barh(top_terms[::-1], top_scores[::-1], color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Mean TF-IDF')
    ax.set_title(cat_name.replace('.', ' / '), fontsize=11)
    ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('Top-10 TF-IDF Terms by Category', fontsize=14)
plt.tight_layout()
plt.show()

## 5. SVD Embeddings from TF-IDF

In [ ]:
# Reduce TF-IDF to 2D with TruncatedSVD
svd_2d = TruncatedSVD(n_components=2, random_state=42)
X_svd_2d = svd_2d.fit_transform(X_tfidf)

# Visual 6: Document embeddings scatter
fig, ax = plt.subplots(figsize=(10, 6))
for cat_id, (cat_name, color) in enumerate(zip(target_names, CATEGORY_COLORS)):
    mask = y_text == cat_id
    ax.scatter(X_svd_2d[mask, 0], X_svd_2d[mask, 1], s=20, alpha=0.5,
               color=color, label=cat_name, edgecolors='none')

ax.set_xlabel(f'SVD Component 1 ({svd_2d.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'SVD Component 2 ({svd_2d.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Document Embeddings: TF-IDF + SVD (2D)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Explained variance (2 components): {svd_2d.explained_variance_ratio_.sum():.1%}')

## 6. Embeddings for Classification

In [ ]:
# Compare classification accuracy: raw TF-IDF vs SVD(50) vs SVD(10)
clf = LogisticRegression(max_iter=1000, random_state=42)

# Raw TF-IDF (1000 features)
scores_raw = cross_val_score(clf, X_tfidf, y_text, cv=5, scoring='accuracy')

# SVD with 50 components
svd_50 = TruncatedSVD(n_components=50, random_state=42)
X_svd50 = svd_50.fit_transform(X_tfidf)
scores_svd50 = cross_val_score(clf, X_svd50, y_text, cv=5, scoring='accuracy')

# SVD with 10 components
svd_10 = TruncatedSVD(n_components=10, random_state=42)
X_svd10 = svd_10.fit_transform(X_tfidf)
scores_svd10 = cross_val_score(clf, X_svd10, y_text, cv=5, scoring='accuracy')

# Visual 7: Accuracy comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))
labels = [f'Raw TF-IDF\n(1000 features)', f'SVD(50)\n(50 features)', f'SVD(10)\n(10 features)']
means = [scores_raw.mean(), scores_svd50.mean(), scores_svd10.mean()]
stds = [scores_raw.std(), scores_svd50.std(), scores_svd10.std()]
colors = [ML_BLUE, ML_PURPLE, ML_ORANGE]

bars = ax.bar(labels, means, yerr=stds, color=colors, alpha=0.7,
              capsize=8, edgecolor='white', linewidth=1.5)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
            f'{m:.3f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Classification Accuracy: Raw TF-IDF vs SVD Embeddings')
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Raw TF-IDF (1000): {scores_raw.mean():.3f} +/- {scores_raw.std():.3f}')
print(f'SVD(50):           {scores_svd50.mean():.3f} +/- {scores_svd50.std():.3f}')
print(f'SVD(10):           {scores_svd10.mean():.3f} +/- {scores_svd10.std():.3f}')

In [ ]:
# Visual 8: t-SNE of SVD(50) embeddings colored by category
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_svd50)

fig, ax = plt.subplots(figsize=(10, 6))
for cat_id, (cat_name, color) in enumerate(zip(target_names, CATEGORY_COLORS)):
    mask = y_text == cat_id
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=20, alpha=0.6,
               color=color, label=cat_name, edgecolors='none')

ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title('t-SNE of Document Embeddings (SVD 50D)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

**Key Takeaways:**
- **One-hot encoding is sparse and captures no similarity** -- dense embeddings place semantically related words close together in vector space
- **Cosine similarity is the standard metric** for comparing word embeddings, revealing cluster structure (bullish vs bearish vs instruments)
- **Vector arithmetic captures relational structure** -- analogies like king-man+woman=queen and stock-equity+debt=bond emerge from good embeddings
- **TF-IDF + SVD is a simple, effective embedding pipeline** -- dimensionality reduction via SVD often preserves classification accuracy while cutting features by 95%+